# 🐍 Master Guide: Metaclasses & Dynamic Type Creation
> **Single-Source Revision Guide**: Covers dynamic classes via `type()`, `__init_subclass__`, custom metaclasses (`__new__` and `__init__`), and class validation.
>
> *Run each cell interactively with `Shift + Enter`.*

---

## 📌 1. Topic Overview
In Python, **classes are instances of metaclasses**. The default metaclass is `type`. Metaprogramming allows code to inspect, generate, and modify classes at definition time.

---
## 🧠 2. CPython Architecture: The Metaclass Hierarchy

```
                    Metaclass (type)
                          │ creates
                          ▼
                     Class (User)
                          │ creates
                          ▼
                  Instance (user_obj)
```

- `isinstance(obj, Class)` is `True`.
- `isinstance(Class, type)` is `True`!
- `type` is its own metaclass: `isinstance(type, type)` is `True`.

---
## 🟢 3. Part 1: Beginner Fundamentals

### Step 1: Creating Classes Dynamically with `type(name, bases, dict)`

In [ ]:
# Normal class:
# class Robot:
#     brand = "CyberCorp"
#     def speak(self): return f"I am a {self.brand} robot!"

# Equivalent dynamic creation via type():
def speak_method(self):
    return f"I am a {self.brand} robot!"

Robot = type("Robot", (object,), {"brand": "CyberCorp", "speak": speak_method})

r = Robot()
print("Dynamic class instance:", r.speak())
print("Class of r            :", type(r))
print("Class of Robot (Meta) :", type(Robot))

---
## 🟡 4. Part 2: Intermediate Features

### Modern Pythonic Alternative: `__init_subclass__` (PEP 487)
In 90% of use cases, you do NOT need full metaclasses! `__init_subclass__` intercepts subclass creation cleanly.

In [ ]:
class PluginBase:
    registry = {}
    
    def __init_subclass__(cls, plugin_name, **kwargs):
        super().__init_subclass__(**kwargs)
        # Automatically registers any subclass upon definition:
        cls.registry[plugin_name] = cls
        print(f"Registered plugin: '{plugin_name}' -> {cls.__name__}")

class AudioPlugin(PluginBase, plugin_name="audio"):
    pass

class VideoPlugin(PluginBase, plugin_name="video"):
    pass

print("Plugin Registry:", PluginBase.registry)

### Custom Metaclasses (`class Meta(type):`)
Enforcing that all class method names must be lowercase.

In [ ]:
class EnforceLowercaseMeta(type):
    def __new__(cls, name, bases, attrs):
        for attr_name in attrs:
            if not attr_name.startswith("__") and callable(attrs[attr_name]):
                if not attr_name.islower():
                    raise TypeError(f"Method '{attr_name}' in class '{name}' must be lowercase!")
        return super().__new__(cls, name, bases, attrs)

# Compliant class:
class ValidService(metaclass=EnforceLowercaseMeta):
    def process_data(self): pass

print("ValidService created successfully.")

# Non-compliant class:
try:
    class InvalidService(metaclass=EnforceLowercaseMeta):
        def ProcessData(self): pass  # Violates rule!
except TypeError as e:
    print("Metaclass validation caught:", e)

---
## 📋 5. Part 3: Quick Revision Cheat Sheet

```
+------------------------------------+---------------------------------------+
| Concept                            | Key Rule                              |
+------------------------------------+---------------------------------------+
| type(name, bases, dict)            | Creates a class dynamically           |
| __init_subclass__                  | Modern, clean hook for subclassing    |
| class Meta(type):                  | Defines a custom metaclass            |
| Meta.__new__                       | Intercepts class object allocation    |
| Python Zen quote                   | 'Metaclasses are deeper magic...'     |
+------------------------------------+---------------------------------------+
```

---
## 🎓 Practical Hands-On & Real-World Examples

### 1. Dynamic Class Generation via `type()` (The 3-Argument `type`)
> In Python, `type` is not only a function to check an object's type (`type(x)`),
> but it is also the default **metaclass** that creates classes dynamically at runtime:
> `type(class_name, tuple_of_bases, dict_of_attributes_and_methods)`

In [ ]:
# Dynamic Class Creation using 3-argument type()
def student_introduce(self):
    return f"Hello, I am {self.name} studying {self.course} at {self.college}."

# Dynamically assembling the class at runtime
StudentClass = type(
    "DynamicStudent",
    (object,),
    {
        "college": "Python Tech Academy",
        "introduce": student_introduce,
        "__init__": lambda self, name, course: setattr(self, "name", name) or setattr(self, "course", course)
    }
)

stu_obj = StudentClass("Harish", "Cloud Architecture")
print("Class Name:   ", StudentClass.__name__)
print("Instance Type:", type(stu_obj))
print("Method Output:", stu_obj.introduce())